# 1. Explore and invoke a serverless function

**Pre-live authoring draft — not ready for participant delivery.**

Run this notebook in a Python kernel from this module's `beginner/` directory.
The offline sections require only Python's standard library. Live sections are
skipped by default; running all cells with the defaults makes no network requests.

By the end, you should be able to trace an HTTP request through the gateway,
identify the greeting input contract, and inspect status, selected headers and
response data without assuming every error is JSON.

The developer supplies function logic. The organiser prepares the runtime,
network routes, protected access and monitoring. A function name prefix is not
an authorisation boundary. Platform edition, builder, registry, authentication,
network isolation and the complete lifecycle remain validation gates.


## 1. Locate the teaching assets — offline

**Purpose:** locate the module and import its helpers without installing packages.
**Location:** this notebook's Python kernel.
**Expected result:** the module path and an offline configuration summary.
**Recovery:** if the path check fails, open the notebook from `beginner/` and
restart the kernel there. Do not substitute paths to another group's workspace.


In [ ]:
from pathlib import Path
import json
import sys

MODULE = Path.cwd().resolve().parent
if not (MODULE / "config/lab.example.json").is_file():
    raise RuntimeError("Open this notebook from the module's beginner directory.")
sys.path.insert(0, str(MODULE / "helpers"))
from lab_config import ConfigurationError, load_config
from invoke import InvocationError, invoke

EXAMPLE_CONFIG = MODULE / "config/lab.example.json"
example = load_config(EXAMPLE_CONFIG, require_live=False)
print("Module:", MODULE)
print("Offline lab label:", example["lab_id"])
print("Live execution enabled in template:", example["live_test_enabled"])


## 2. Trace the request — offline

The intended path is **Python or curl → HTTPS gateway → function runtime →
handler → HTTP response**. The wrapper parses the request before the handler.
Malformed JSON can therefore fail before your function logic executes.

Read the real greeting source below. It accepts POST, expects an object with a
string `name`, trims that string, and requires a non-empty result of at most 80
JavaScript UTF-16 code units. Python string length is not an equivalent check
for all Unicode characters.

**Checkpoint:** identify the success response, the invalid-name response and
the method-not-allowed response. Printing source is not executing the handler.


In [ ]:
source = (MODULE / "examples/greeting/handler.js").read_text(encoding="utf-8")
print(source)
print("Example valid request:", json.dumps({"name": "Ada"}))
print("Expected handler success:", json.dumps({
    "message": "Hello Ada, your function is live!"
}))


## 3. Verify that the teaching template fails closed — offline

**Purpose:** distinguish authoring configuration from permission to run a live lab.
**Expected result:** the template is rejected for live execution.
A configuration flag is a safeguard, not proof of organiser approval or isolation.


In [ ]:
try:
    load_config(EXAMPLE_CONFIG)
except ConfigurationError as error:
    print("Expected live-configuration rejection:", error)
else:
    raise RuntimeError("The example configuration unexpectedly permits live execution.")


### 3.1 Practise response inspection without a gateway

**Purpose:** separate HTTP status, response format and transport failure.
**Location:** this notebook's Python kernel; no requests are sent.
**Action:** predict what the cell below prints, then run it. It uses the
real response class with invented teaching inputs, not recorded responses.
**Expected result:** status 400 with a parsed `invalid_name` error; a
second status 400 with HTML content type; and a safely caught JSON error.

Status 400 alone does not identify the rejecting component or response
format. Inspect selected headers and handle parsing failure; even a JSON
content type is not proof that the body parses. A transport failure is
different: the invocation helper raises an error instead of returning a
usable response. This exercise does not simulate an actual transport.

**Checkpoint:** why can both examples have status 400 but only one parse
as JSON? Why should raw parser-error bodies not be printed?
**Recovery:** rerun the imports in section 1 if a name is missing. Do not
enable live execution to resolve an offline exercise error.


In [ ]:
from invoke import InvocationResult

synthetic_json_error = InvocationResult(
    status=400,
    headers={"content-type": "application/json"},
    body='{"error": "invalid_name"}',
)
synthetic_html_error = InvocationResult(
    status=400,
    headers={"content-type": "text/html; charset=utf-8"},
    body="<html><body>Synthetic Bad Request</body></html>",
)
print("SYNTHETIC RESPONSE PRACTICE — NO REQUESTS SENT")
print("JSON example status:", synthetic_json_error.status)
print("JSON example headers:", synthetic_json_error.headers)
parsed_synthetic_error = synthetic_json_error.json()
assert parsed_synthetic_error == {"error": "invalid_name"}
print("Synthetic JSON error:", parsed_synthetic_error)
print("HTML example status:", synthetic_html_error.status)
print("HTML example content type:", synthetic_html_error.headers["content-type"])
print("Raw HTML body intentionally not displayed.")
synthetic_parse_rejected = False
try:
    synthetic_html_error.json()
except InvocationError as error:
    synthetic_parse_rejected = True
    print("Expected safe parsing failure:", error)
else:
    raise RuntimeError("Synthetic HTML unexpectedly parsed as JSON.")


## 4. Live execution gate

**STOP unless the organiser has validated the assigned disposable lab and its
invocation access.** No live gateway behaviour has been validated in this draft.

The organiser supplies `config/lab.local.json` through the agreed channel; it is
ignored by Git and must contain no passwords or tokens. Do not enable live access
by editing the example template or bypassing TLS checks. This helper has no
credential support: if the gateway requires authentication, stop until the
approved authentication mechanism has been implemented and tested.

Only after that validation, change `RUN_LIVE` below to `True` and rerun this cell.
Keep it `False` for offline review. This cell checks configuration, not readiness.


In [ ]:
RUN_LIVE = False
LOCAL_CONFIG = MODULE / "config/lab.local.json"
if RUN_LIVE:
    live_config = load_config(LOCAL_CONFIG)
    print("Configuration accepted; platform readiness is not established by this check.")
else:
    print("SKIPPED: live execution is disabled in this notebook.")


## 5. Invoke the prepared greeting — gated live action

**Purpose:** invoke the organiser-prepared function without building or deploying.
**Location:** the assigned lab's Python kernel.
**Expected candidate result:** status 200 and the greeting JSON shown below.
**Verification:** assertions check the observed response; failures are evidence
of a mismatch, not permission to change platform settings.

**Recovery:** stop on TLS, routing or access failure and contact the organiser.
Do not print credentials or disable certificate verification. The helper refuses
redirects, applies a socket timeout, bounds response size, and does not retry.
The socket timeout is not a strict total wall-clock deadline.


In [ ]:
if RUN_LIVE:
    result = invoke(LOCAL_CONFIG, "prepared_greeting", {"name": "Ada"})
    print("HTTP status:", result.status)
    print("Selected headers:", result.headers)
    print("JSON:", result.json())
    assert result.status == 200, "Unexpected status; stop and report the mismatch."
    assert result.json() == {"message": "Hello Ada, your function is live!"}
else:
    print("SKIPPED: prepared greeting invocation.")


## 6. Inspect predictable handler errors — gated live actions

Send only the following small synthetic requests to your assigned prepared
function. Candidate handler expectations are 400 for invalid names or arrays,
and 405 with `Allow: POST` for GET. Actual gateway behaviour still needs rehearsal.
These are four requests, with no retries.


In [ ]:
if RUN_LIVE:
    cases = [
        ({"name": ""}, "POST", 400, {"error": "invalid_name"}),
        ({"name": 7}, "POST", 400, {"error": "invalid_name"}),
        ([], "POST", 400, {"error": "expected_json_object"}),
        (None, "GET", 405, {"error": "method_not_allowed"}),
    ]
    for payload, method, expected_status, expected_body in cases:
        result = invoke(LOCAL_CONFIG, "prepared_greeting", payload, method=method)
        print(method, result.status, result.headers, result.json())
        assert result.status == expected_status
        assert result.json() == expected_body
        if method == "GET":
            assert result.headers.get("allow") == "POST"
else:
    print("SKIPPED: handler error requests.")


## 7. Malformed JSON is a different failure path — gated live action

The inspected direct HTTP wrapper returned status 400 with HTML for malformed
JSON before the handler ran. Production mode suppressed client-facing parser
stack details, but parser diagnostics and malformed input were still observed
in server logs. This does not establish a safe logging policy or gateway result.

Use only this synthetic input, never personal information or credentials.
Inspect status, content type and bounded body length; do not blindly call `.json()`
or display the raw error body. Record any mismatch for organiser review.


In [ ]:
if RUN_LIVE:
    result = invoke(LOCAL_CONFIG, "prepared_greeting", b"{broken")
    print("Observed status:", result.status)
    print("Observed content type:", result.headers.get("content-type"))
    print("Response characters:", len(result.body))
    print("Raw error body intentionally not displayed.")
else:
    print("SKIPPED: malformed JSON request.")


## 8. Review and next steps

- Which layer rejects malformed JSON, and which rejects an empty name?
- Why does changing a request name not require rebuilding the function image?
- Why are a successful request and a function prefix insufficient evidence of
  cross-group isolation?
- What additional evidence distinguishes handler tests, wrapper tests and a
  complete gateway/runtime rehearsal?

This notebook changes no deployed code and deletes no resources. Do not remove
the organiser-prepared function. Notebook 2 will cover owned function source and
candidate manifest preparation; its live build/push/deploy workflow remains gated.

**Draft limitation:** the planned curl exercise is not yet included. It must use
the same validated route and approved authentication mechanism; do not improvise
credential-bearing shell commands. This is not a complete delivery-ready notebook.
